Exame base BI + PID KLINGO

In [16]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

# ==========================================
# 1. CONFIGURAÇÕES E DICIONÁRIOS GLOBAIS
# ==========================================
CAMINHO_PASTA = r"C:\Users\user\Downloads\Script_Exames_HV"
ARQUIVO_EXAME = os.path.join(CAMINHO_PASTA, "EXAME.xlsx")
ARQUIVO_PID = os.path.join(CAMINHO_PASTA, "PID KLINGO.xlsx")
ARQUIVO_FINAL = os.path.join(CAMINHO_PASTA, "Consolidado_Hospital_Visao.xlsx")

MESES_PT = {
    1: "JANEIRO", 2: "FEVEREIRO", 3: "MARÇO", 4: "ABRIL",
    5: "MAIO", 6: "JUNHO", 7: "JULHO", 8: "AGOSTO",
    9: "SETEMBRO", 10: "OUTUBRO", 11: "NOVEMBRO", 12: "DEZEMBRO"
}

MAPA_NOMES = {
    "EDIVANIA LEITE": "EDIVANIA PEREIRA", "GABRIELLA FELIX": "GABRIELLA ALVES",
    "ISABELLA WANDERLEY": "ISABELLA W QUEIROGA", "JOAO VITOR": "JOAO VITOR BRUSQUI",
    "JOSE CARLOS": "JOSE CARLOS EVANGELISTA", "LUCIANA FREITAS": "LUCIANA OLIVEIRA",
    "MARIA CLARA": "MARIA CLARA PALITOT", "MARIANA GADELHA": "MARIANA MELO GADELHA",
    "MAYANNA DANTAS": "MAYANNA PINTO DANTA", "PATRICIA CHERMILLA": "PATRICIA CHERMILA",
    "RAIMUNDO OLIVEIRA": "RAIMUNDO DE OLIVEIRA", "ROBERTA FERNANDES": "ROBERTA FERNANDA",
    "TASSIA LIMA": "TASSIA OLIVEIRA", "TATIANA LUCENA": "TATIANA L OLIVEIRA"
}

# ==========================================
# 2. FUNÇÕES DE SUPORTE
# ==========================================
def corrigir_texto(texto):
    if pd.isna(texto): return texto
    texto_corrigido = str(texto).upper()
    mapa_correcao = {
        "Ã‡": "Ç", "Ãƒ": "Ã", "ÃƑ": "Ã", "Ã“": "Ó", "Ã‰": "É",
        "Ã‚": "Â", "ÃŠ": "Ê", "Ã": "Í", "Ã•": "Õ", "Ãš": "Ú", "Ã€": "À", "A‡A": "ÇÃ"
    }
    for erro, acerto in mapa_correcao.items():
        texto_corrigido = texto_corrigido.replace(erro, acerto)
    return texto_corrigido.strip()

def classificar_procedimento(nome, grupo):
    nome = str(nome).upper()
    if grupo.upper() == 'EXAMES':
        if any(x in nome for x in ['RETINA', 'OCT', 'ANGIO', 'FOTOCOAGULA', 'FUNDO']):
            return 'Exames - Retina' if 'NERVO' not in nome else 'Exames - Glaucoma / Nervo Óptico'
        elif 'TONOMETRIA' in nome: return 'Exames - Glaucoma'
        elif any(x in nome for x in ['CÓRNEA', 'CERATOMETRIA', 'PAQUIMETRIA', 'ESPECULAR']): return 'Exames - Córnea'
        elif any(x in nome for x in ['BIOMETRIA', 'CAPSULOTOMIA', 'YAG']): return 'Exames - Catarata'
    return grupo.capitalize()

def tratar_data_segura(chave):
    if pd.isna(chave) or str(chave).strip() == '':
        return 2025, "DESCONHECIDO"

    if hasattr(chave, 'year') and hasattr(chave, 'month'):
        ano = chave.year if chave.year >= 2000 else 2025
        return ano, MESES_PT.get(chave.month, "DESCONHECIDO")

    if isinstance(chave, (int, float)):
        if chave > 10000: 
            try:
                dt = pd.to_datetime(chave, unit='D', origin='1899-12-30')
                ano = dt.year if dt.year >= 2000 else 2025
                return ano, MESES_PT.get(dt.month, "DESCONHECIDO")
            except: pass
        return 2025, "DESCONHECIDO"

    texto = str(chave).strip().lower()
    mapa_str = {'jan': 'JANEIRO', 'fev': 'FEVEREIRO', 'mar': 'MARÇO', 'abr': 'ABRIL', 'mai': 'MAIO', 'jun': 'JUNHO', 'jul': 'JULHO', 'ago': 'AGOSTO', 'set': 'SETEMBRO', 'out': 'OUTUBRO', 'nov': 'NOVEMBRO', 'dez': 'DEZEMBRO'}

    if '/' in texto:
        partes = texto.split('/')
        if partes[0][:3] in mapa_str:
            mes = mapa_str[partes[0][:3]]
            try:
                ano = int("20" + partes[1]) if len(partes[1]) == 2 else int(partes[1])
            except:
                ano = 2025
            return ano, mes

    mes = mapa_str.get(texto[:3], "DESCONHECIDO")
    return 2025, mes

# ==========================================
# 3. EXTRAÇÃO DAS DUAS BASES
# ==========================================
def extrair_klingo():
    print("Lendo PID KLINGO...")
    if not os.path.exists(ARQUIVO_PID): return pd.DataFrame()

    xls = pd.ExcelFile(ARQUIVO_PID)
    abas_medicos = [aba for aba in xls.sheet_names if aba.upper() != 'PID GERAL']
    dados_fatos = []

    for aba in abas_medicos:
        df = pd.read_excel(xls, sheet_name=aba, header=None)
        medico = MAPA_NOMES.get(aba.strip().upper(), aba.strip().upper())
        
        for bloco in ['CONSULTAS', 'EXAMES', 'CIRURGIAS', 'LENTES', 'PROCEDIMENTOS']:
            l, c = None, None
            for i in range(df.shape[0]):
                for j in range(df.shape[1]):
                    if str(df.iloc[i, j]).strip().upper() == bloco:
                        l, c = i, j
                        break
                if l is not None: break
            
            if l is None: continue
            
            col_meses = {}
            for col_idx in range(c + 1, df.shape[1]):
                val = df.iloc[l, col_idx]
                if pd.isna(val) or str(val).strip().upper() in ['TOTAL GERAL', 'TOTAL', 'NAN', 'NAT', '']: break
                col_meses[col_idx] = val

            for i in range(l + 2, df.shape[0]):
                proc_raw = df.iloc[i, c]
                if pd.isna(proc_raw) or str(proc_raw).strip().upper() in ['TOTAL GERAL', 'TOTAL', 'NAN', 'NAT', '']: break
                
                procedimento = corrigir_texto(proc_raw)
                for col_idx, data_chave in col_meses.items():
                    qtd = df.iloc[i, col_idx]
                    
                    if pd.notna(qtd) and str(qtd).strip() != '':
                        try:
                            qtd_num = float(qtd)
                            if qtd_num != 0:
                                dados_fatos.append({
                                    'Chave_Data': data_chave, 'Médico': medico,
                                    'Grupo_Geral': bloco.capitalize(), 
                                    'Procedimento': procedimento,
                                    'Quantidade': qtd_num
                                })
                        except (ValueError, TypeError):
                            pass

    return pd.DataFrame(dados_fatos)

def extrair_exame_bi():
    print("Lendo EXAME_Base_BI...")
    if not os.path.exists(ARQUIVO_EXAME): return pd.DataFrame()

    df_raw = pd.read_excel(ARQUIVO_EXAME, sheet_name="GERAL", header=None)
    anos, meses = df_raw.iloc[0].ffill(), df_raw.iloc[1].ffill()
    metricas_originais = df_raw.iloc[2]
    
    metricas_limpas = []
    for m in metricas_originais:
        m_str = str(m).strip().upper()
        if m_str in ['CONS', 'CONSULTAS']: metricas_limpas.append('Consultas')
        elif m_str in ['EX', 'EXAMES']: metricas_limpas.append('Exames')
        elif m_str in ['CIR', 'CIRURGIAS']: metricas_limpas.append('Cirurgias')
        else: metricas_limpas.append(str(m).strip().capitalize())
        
    dados_bi = []
    for i in range(3, len(df_raw)):
        medico = MAPA_NOMES.get(str(df_raw.iloc[i, 0]).strip().upper(), str(df_raw.iloc[i, 0]).strip().upper())
        if pd.isna(df_raw.iloc[i, 0]) or medico == "NAN": continue
        
        for j in range(1, len(df_raw.columns)):
            val = df_raw.iloc[i, j]
            if pd.isna(val) or str(val).strip() == '': continue
            
            try:
                val_num = float(val)
                ano_num = int(float(anos[j]))
                if ano_num < 2000: ano_num = 2025 # Trava anti-1970
                
                dados_bi.append({
                    'Ano': ano_num, 'Mês': str(meses[j]).strip().upper(),
                    'Médico': medico, 'Métrica': metricas_limpas[j], 'Quantidade': val_num
                })
            except (ValueError, TypeError):
                pass
    
    if not dados_bi: return pd.DataFrame()
        
    df = pd.DataFrame(dados_bi).pivot_table(index=['Ano', 'Mês', 'Médico'], 
                                           columns='Métrica', values='Quantidade', aggfunc='sum').reset_index()
    return df.fillna(0)

# ==========================================
# 4. EXECUÇÃO E CONSOLIDAÇÃO INTELIGENTE
# ==========================================
def consolidar_tudo():
    print("Iniciando a unificação das bases...")
    
    # 1. Processar Klingo (Detalhada)
    df_detalhada = extrair_klingo()
    if df_detalhada.empty:
        print("Erro: Nenhum dado extraído do Klingo.")
        return
        
    df_detalhada[['Ano', 'Mês']] = df_detalhada.apply(lambda row: pd.Series(tratar_data_segura(row['Chave_Data'])), axis=1)
    df_detalhada['Categoria'] = df_detalhada.apply(lambda r: classificar_procedimento(r['Procedimento'], r['Grupo_Geral']), axis=1)
    
    # Gerar a versão BI do Klingo
    df_klingo_bi = df_detalhada.groupby(['Ano', 'Mês', 'Médico', 'Grupo_Geral'])['Quantidade'].sum().unstack(fill_value=0).reset_index()
    df_klingo_bi['Origem'] = 'KLINGO'

    # 2. Processar Exame
    df_exame_bi = extrair_exame_bi()
    if not df_exame_bi.empty:
        df_exame_bi['Origem'] = 'EXAME'
    
    # 3. UNIÃO COM PRIORIDADE KLINGO (Resolve o erro do 51 + 3 = 54)
    colunas_padrao = ['Consultas', 'Exames', 'Cirurgias', 'Lentes', 'Procedimentos']
    for col in colunas_padrao:
        if col not in df_klingo_bi.columns: df_klingo_bi[col] = 0
        if not df_exame_bi.empty and col not in df_exame_bi.columns: df_exame_bi[col] = 0

    df_klingo_idx = df_klingo_bi.set_index(['Ano', 'Mês', 'Médico'])
    
    if not df_exame_bi.empty:
        df_exame_idx = df_exame_bi.set_index(['Ano', 'Mês', 'Médico'])
        # Transforma 0 em NaN temporariamente para a sobreposição funcionar
        df_klingo_idx = df_klingo_idx.replace(0, np.nan)
        df_exame_idx = df_exame_idx.replace(0, np.nan)
        
        # combine_first: Mantém o KLINGO, preenche buracos com EXAME
        df_consolidada = df_klingo_idx.combine_first(df_exame_idx).fillna(0).reset_index()
    else:
        df_consolidada = df_klingo_idx.fillna(0).reset_index()
            
    df_consolidada['Procedimentos e Lentes'] = df_consolidada['Procedimentos'] + df_consolidada['Lentes']
    
    # 4. Cálculo de Porcentagens
    colunas_finais = ['Consultas', 'Exames', 'Cirurgias', 'Lentes', 'Procedimentos', 'Procedimentos e Lentes']
    for col in colunas_finais:
        total_mes = df_consolidada.groupby(['Ano', 'Mês'])[col].transform('sum')
        df_consolidada[f'% de {col}'] = (df_consolidada[col] / total_mes).replace([np.inf, -np.inf], 0).fillna(0)

    # 5. Ordenar Colunas
    ordem_final_bi = ['Ano', 'Mês', 'Médico', 'Origem', 
                   'Consultas', '% de Consultas', 
                   'Exames', '% de Exames', 
                   'Cirurgias', '% de Cirurgias', 
                   'Lentes', '% de Lentes', 
                   'Procedimentos', '% de Procedimentos', 
                   'Procedimentos e Lentes', '% de Procedimentos e Lentes']
                   
    df_consolidada = df_consolidada[ordem_final_bi]

    # 6. Salvar as duas abas
    with pd.ExcelWriter(ARQUIVO_FINAL) as writer:
        df_consolidada.to_excel(writer, sheet_name='BI_Consolidado', index=False)
        df_detalhada[['Ano', 'Mês', 'Médico', 'Grupo_Geral', 'Categoria', 'Procedimento', 'Quantidade']].to_excel(writer, sheet_name='Detalhado_Klingo', index=False)
    
    print(f"\n✅ SUCESSO! O arquivo final (Klingo blindado + Exame) foi gerado em:\n{ARQUIVO_FINAL}")

if __name__ == "__main__":
    consolidar_tudo()

Iniciando a unificação das bases...
Lendo PID KLINGO...
Lendo EXAME_Base_BI...

✅ SUCESSO! O arquivo final (Klingo blindado + Exame) foi gerado em:
C:\Users\user\Downloads\Script_Exames_HV\Consolidado_Hospital_Visao.xlsx
